## GazeOnlyGAT: components overview

In [1]:
import only_gaze_confs as conf

/store/alutfi/py_venvs/gat2/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


choosing least busy GPU...
running on cuda:1


With the gaze patch features and encodings aggregated into graph nodes, we can feed it through our network. We'll go through each component.

Fetch sample batch

In [2]:
import torch
from gaze_datasets.reflacx import ReflacxGazePatchDataset
from gaze_datasets.dataset_tools import gaze_patch_dataset_collate
from feature_extraction.extractors import GazePatchFeatureExtractor
from node_encodings.sequence import GazeSinCosPosEncoding

gaze_dataset_pth = conf.reflacx_gaze_dataset_pth
desired_label = 'Atelectasis'
ds = ReflacxGazePatchDataset(gaze_dataset_pth, desired_label)
splits, fixations, patches, labels = gaze_patch_dataset_collate((ds[33], ds[34], ds[0]))

feat_ext = GazePatchFeatureExtractor(conf.fe_tail_feat_ext,
                                     conf.fe_state_dict_pth,
                                     conf.fe_channels,
                                     conf.fe_act,
                                     conf.fe_norm,
                                     conf.fe_bias,
                                     conf.fe_w_init_f,
                                     conf.device)
feats = feat_ext(patches.to(conf.device))

max_len = conf.reflacx_max_len
seq_encoding_featcount = conf.seq_encoding_feat_count

seq_encoding = GazeSinCosPosEncoding(max_len, seq_encoding_featcount)
seqs = seq_encoding(splits, adjust_dim=True).to(conf.device)
fixations = fixations.to(conf.device)

nodes = torch.cat([fixations, seqs, feats], 1).float()
nodes.shape, sum([x.shape[1]for x in (seqs, fixations, feats)])

(torch.Size([255, 147, 1, 1]), 147)

In [3]:
ns = nodes.squeeze(-1).squeeze(-1).permute((1, 0))

### GAT Layer
<a id="gat_layer"></a>

The first component is the Graph Attention Convolution Layer.

In [4]:
from model.gat_conv import CausalGATConv

set number of attention heads for the layer

In [5]:
att_heads = 4

In [6]:
gcnn = CausalGATConv(ns.shape[0],
                     ns.shape[0],
                     att_heads,
                     device=conf.device)

In [7]:
conv_nodes = gcnn((ns, splits))
conv_nodes.shape

torch.Size([147, 255])

### XYTGATLayer

a gat layer that has attention mechanisms that look only into pair of nodex xy coord and pairs of time duration

In [8]:
from xyt_gat_conv import CausalXYTGATConv
from gaze_datasets.dataset_tools import get_xyt_pair_matrices

In [9]:
xy_mats, t_mats = get_xyt_pair_matrices(fixations, splits)

In [10]:
gcnn = CausalXYTGATConv(ns.shape[0],
                        ns.shape[0],
                        8,
                        device=conf.device)

In [11]:
gcnn

CausalXYTGATConv(
  (att_scores): Conv2d(294, 8, kernel_size=(1, 1), stride=(1, 1))
  (xy_scores): Conv2d(4, 2, kernel_size=(1, 1), stride=(1, 1))
  (t_scores): Conv2d(2, 1, kernel_size=(1, 1), stride=(1, 1))
  (node_update): Conv1d(1617, 147, kernel_size=(1,), stride=(1,))
  (norm): GraphNorm(147)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [12]:
conv_nodes = gcnn((ns, splits, xy_mats, t_mats))
conv_nodes.shape

torch.Size([147, 255])

### GazeGrapher
<a id="grapher"></a>

GazeGrapher is a module that aggregates several GAT layers in sequence, performing all graph convolutions before graph readout and prediction.

PyTorch's native nn.Sequential does not keep residual tensors between its internal steps, so we need to extend it. Graph readout and predictions will need residual, intermediary, graphs

In [13]:
from residual_sequential import MarkovResidualSequential, DenseResidualSequential

We need input sizes for each of the graphs GAT Layers, and the residual layers' indices and GAT conv class of choice

In [14]:
layers = 4
gat_channels = 256 # input size and 4 GAT convolution layers
residual_gat_levels = [False, True, True, True]
att_heads = [16, 8, 8, 8]
output_gat_levels = None
gat_conv = CausalGATConv

In [15]:
from grapher import GazeGrapher, XYTGazeGrapher

In [16]:
from torch import nn
from torch_geometric.nn.norm import GraphNorm
import torch.nn.functional as F

from model.gat_conv import CausalGATConv

In [17]:
grapher = GazeGrapher(layers=layers,
                      output_gat_levels=output_gat_levels,
                      residual_gat_levels=residual_gat_levels,
                      att_heads=att_heads,
                      gat_conv_args={'channels': gat_channels,
                                     'act_att': F.leaky_relu,
                                     'act_gat': F.elu,
                                     'norm': GraphNorm,
                                     'dropout': 0.1,
                                     'bias': True,
                                     'w_init_f': lambda w: nn.init.xavier_uniform_(w)},
                      device=conf.device)
grapher

GazeGrapher(
  (gats): MarkovResidualSequential(
    (GAT_-1): CausalGATConv(
      (att_scores): Conv2d(512, 16, kernel_size=(1, 1), stride=(1, 1))
      (node_update): Conv1d(4096, 256, kernel_size=(1,), stride=(1,))
      (norm): GraphNorm(256)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (GAT_0): CausalGATConv(
      (att_scores): Conv2d(512, 16, kernel_size=(1, 1), stride=(1, 1))
      (node_update): Conv1d(4096, 256, kernel_size=(1,), stride=(1,))
      (norm): GraphNorm(256)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (GAT_1): CausalGATConv(
      (att_scores): Conv2d(512, 16, kernel_size=(1, 1), stride=(1, 1))
      (node_update): Conv1d(4096, 256, kernel_size=(1,), stride=(1,))
      (norm): GraphNorm(256)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (GAT_2): CausalGATConv(
      (att_scores): Conv2d(512, 16, kernel_size=(1, 1), stride=(1, 1))
      (node_update): Conv1d(4096, 256, kernel_size=(1,), stride=(1,))
      (norm): GraphNorm(256)

In [18]:
expand_dims = nn.Conv1d(ns.shape[0], gat_channels, 1).to(conf.device)
ens = expand_dims(ns)
ens.shape

torch.Size([256, 255])

In [19]:
graphs = grapher((ens, splits)) # output is all selected residual graphs by residual_gat_levels
[g.shape for g in graphs]

[torch.Size([256, 255]),
 torch.Size([256, 255]),
 torch.Size([256, 255]),
 torch.Size([256, 255])]

In [20]:
grapher = XYTGazeGrapher(layers=layers,
                         output_gat_levels=output_gat_levels,
                         residual_gat_levels=residual_gat_levels,
                         att_heads=att_heads,
                         gat_conv_args={'channels': gat_channels,
                                        'xy_heads_ratio': 0.2,
                                        't_heads_ratio': 0.1,
                                        'act_att': F.leaky_relu,
                                        'act_gat': F.elu,
                                        'norm': GraphNorm,
                                        'dropout': 0.1,
                                        'bias': True,
                                        'w_init_f': lambda w: nn.init.xavier_uniform_(w)},
                         device=conf.device)
grapher

XYTGazeGrapher(
  (gats): MarkovResidualSequential(
    (GAT_-1): CausalXYTGATConv(
      (att_scores): Conv2d(512, 16, kernel_size=(1, 1), stride=(1, 1))
      (xy_scores): Conv2d(4, 3, kernel_size=(1, 1), stride=(1, 1))
      (t_scores): Conv2d(2, 2, kernel_size=(1, 1), stride=(1, 1))
      (node_update): Conv1d(5376, 256, kernel_size=(1,), stride=(1,))
      (norm): GraphNorm(256)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (GAT_0): CausalXYTGATConv(
      (att_scores): Conv2d(512, 16, kernel_size=(1, 1), stride=(1, 1))
      (xy_scores): Conv2d(4, 3, kernel_size=(1, 1), stride=(1, 1))
      (t_scores): Conv2d(2, 2, kernel_size=(1, 1), stride=(1, 1))
      (node_update): Conv1d(5376, 256, kernel_size=(1,), stride=(1,))
      (norm): GraphNorm(256)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (GAT_1): CausalXYTGATConv(
      (att_scores): Conv2d(512, 16, kernel_size=(1, 1), stride=(1, 1))
      (xy_scores): Conv2d(4, 3, kernel_size=(1, 1), stride=(1, 1))
    

In [21]:
expand_dims = nn.Conv1d(ns.shape[0], gat_channels, 1).to(conf.device)
ens = expand_dims(ns)
ens.shape

torch.Size([256, 255])

In [22]:
graphs = grapher((ens, splits, fixations)) # output is all selected residual graphs by residual_gat_levels
[g.shape for g in graphs]

[torch.Size([256, 255]),
 torch.Size([256, 255]),
 torch.Size([256, 255]),
 torch.Size([256, 255])]

In [23]:
graphs = grapher((ens, splits, fixations)) # output is all selected residual graphs by residual_gat_levels
[g.shape for g in graphs]

[torch.Size([256, 255]),
 torch.Size([256, 255]),
 torch.Size([256, 255]),
 torch.Size([256, 255])]

### Readout
<a id="readout"></a>

Since we are dealing with complete causal graphs --each node sees ALL of its predecessors and itself-- we developed a causal graph readout. All information is aggregated by the last node at each convolution. So, we take the last node of each graph selected by residual_gat_levels, concatenate them, and run them through linear layers, outputting the final predictions.

In [24]:
from readout import CausalReadout

In [25]:
sum([g.shape[0] for g in graphs])

1024

In [26]:
ro_channels=conf.ro_channels
ro_channels[0] = sum([g.shape[0] for g in graphs])

In [27]:
causal_ro = CausalReadout(channels=ro_channels,
                          dropout=conf.ro_dropout,
                          device=conf.device)

In [28]:
causal_ro

CausalReadout(
  (act): Identity()
  (mlp): Sequential(
    (fc0): Conv1d(1024, 512, kernel_size=(1,), stride=(1,))
    (norm1): GroupNorm(1, 512, eps=1e-05, affine=True)
    (act1): GELU(approximate='none')
    (drop1): Dropout(p=0.15, inplace=False)
    (fc1): Conv1d(512, 256, kernel_size=(1,), stride=(1,))
    (norm2): GroupNorm(1, 256, eps=1e-05, affine=True)
    (act2): GELU(approximate='none')
    (drop2): Dropout(p=0.15, inplace=False)
    (fc2): Conv1d(256, 128, kernel_size=(1,), stride=(1,))
    (norm3): GroupNorm(1, 128, eps=1e-05, affine=True)
    (act3): GELU(approximate='none')
    (drop3): Dropout(p=0.15, inplace=False)
    (fc3): Conv1d(128, 64, kernel_size=(1,), stride=(1,))
    (norm4): GroupNorm(1, 64, eps=1e-05, affine=True)
    (act4): GELU(approximate='none')
    (drop4): Dropout(p=0.15, inplace=False)
    (prediction): Conv1d(64, 1, kernel_size=(1,), stride=(1,))
  )
)

In [29]:
predictions = causal_ro((graphs, splits))
predictions.shape, labels.shape

(torch.Size([3]), torch.Size([3]))

### Whole pipeline
<a id="full"></a>

Since we now have all individual components, we are ready to run the full model.

In [30]:
from gaze_only_gat import GazeOnlyGAT

In [31]:
gat_conv_args={'channels': gat_channels,
               'att_heads': 8,
               'act_att': F.leaky_relu,
               'act_gat': F.elu,
               'norm': GraphNorm,
               'dropout': 0.1,
               'bias': True,
               'w_init_f': lambda w: nn.init.xavier_uniform_(w)}

In [32]:
model = GazeOnlyGAT(feat_extractor=feat_ext,
                    encoder=GazeSinCosPosEncoding(386, 16),
                    gat_conv=CausalGATConv,
                    gat_conv_args=gat_conv_args,
                    ro_hidden_features=ro_channels[1:],
                    device=conf.device)

In [33]:
predictions = model((splits,
                     fixations.to(conf.device),
                     patches.to(conf.device),
                     labels.to(conf.device)))
predictions.shape, labels.shape

(torch.Size([3]), torch.Size([3]))

In [34]:
xyt_gat_conv_args={'channels': gat_channels,
                   'att_heads': 8,
                   'xy_heads_ratio': 0.2,
                   't_heads_ratio': 0.1,
                   'act_att': F.leaky_relu,
                   'act_gat': F.elu,
                   'norm': GraphNorm,
                   'dropout': 0.1,
                   'bias': True,
                   'w_init_f': lambda w: nn.init.xavier_uniform_(w)}

In [35]:
model = GazeOnlyGAT(feat_extractor=feat_ext,
                    encoder=GazeSinCosPosEncoding(386, 16),
                    gat_conv=CausalXYTGATConv,
                    gat_conv_args=xyt_gat_conv_args,
                    ro_hidden_features=ro_channels[1:],
                    device=conf.device)

In [36]:
predictions = model((splits,
                     fixations.to(conf.device),
                     patches.to(conf.device),
                     labels.to(conf.device)))
predictions.shape, labels.shape

(torch.Size([3]), torch.Size([3]))